In [1]:
# ЯЧЕЙКА 1
# Импорты. Только то, что нужно для LightGBM.

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import lightgbm as lgb


In [2]:
# ЯЧЕЙКА 2
# Загрузка sample-датасета с таргетом.
# Используем ТОТ ЖЕ файл, что и для CatBoost.

DATASET_PATH = "../data/processed/dataset_with_target_sample.pkl"

df = pd.read_pickle(DATASET_PATH)

df.shape


(300000, 180)

In [4]:
# ЯЧЕЙКА 3
# Базовые проверки.

assert "flag" in df.columns
assert "id" in df.columns

df["flag"].value_counts(normalize=True)


flag
0    0.967097
1    0.032903
Name: proportion, dtype: float64

In [5]:
# ЯЧЕЙКА 4
# Формируем X / y.
# id исключаем, OHE-категории (если остались) тоже исключаем.

drop_cols = ["id", "flag"]

X = df.drop(columns=drop_cols)
y = df["flag"]

X.shape


(300000, 178)

In [6]:
# ЯЧЕЙКА 5
# Train / Validation split со стратификацией.

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train.shape, X_val.shape


((240000, 178), (60000, 178))

In [7]:
# ЯЧЕЙКА 6
# Настройка class imbalance.

pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = neg / pos

scale_pos_weight


29.39128783082183

In [8]:
# ЯЧЕЙКА 7
# Обучение LightGBM.
# Параметры подобраны под агрегированные risk-фичи.

model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    min_child_samples=200,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary",
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(stopping_rounds=200, verbose=True)
    ],
)


[LightGBM] [Info] Number of positive: 7897, number of negative: 232103
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020450 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4292
[LightGBM] [Info] Number of data points in the train set: 240000, number of used features: 144
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.032904 -> initscore=-3.380698
[LightGBM] [Info] Start training from score -3.380698
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.671328	valid_0's binary_logloss: 0.145009


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.03
,n_estimators,2000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,200


In [9]:
# ЯЧЕЙКА 8
# ROC-AUC на валидации.

proba_val = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, proba_val)

roc_auc


0.6713275493908314

In [3]:
# ЯЧЕЙКА 9
# Важности признаков — контроль, что модель видит сигнал.

feat_imp = (
    pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
)

feat_imp.head(20)


NameError: name 'X' is not defined